# Publish to PostgreSQL

## Objective

Send a completed run to PostgreSQL and safely repeat the publication.

Run notebooks 2 through 4 first. This writes saved forecasts and scores, including every retained tuning window, without fitting or rescoring. You need write access to your chosen destination. These adapters publish results; they do not add source-data ingestion from this database.

Every row carries study ID/hash, workflow run ID, stage, origin and horizon. Tuning rows also carry search ID, trial ID, attempt ID and zero-based window index. Final validation/test rows have null tuning identifiers. Join using the complete identity and null-safe comparisons. Recorded scores remain in `evidence_json`.

Install `autoforecast[postgres]`. Put a connection string in `AUTOFORECAST_POSTGRES_DSN` outside the notebook. Use an existing database with a `public` schema and tables reserved for this example. The caller owns credentials and the connection.


## Step 1: Choose a place for the example files

We will keep data and results outside the library folder. The printed path shows where to find them. Set the environment variable only if you want a different folder.


We also turn on progress messages, so we can see which step is running.


In [ ]:
# Import the tools used in this step.
import os
from pathlib import Path

from autoforecast import configure_logging

# Choose the workspace without changing the library checkout.
work = (
    Path(os.environ.get("AUTOFORECAST_TUTORIAL_DIR", Path.home() / "autoforecast-tutorials"))
    .expanduser()
    .resolve()
)
# Create the folder if this is your first visit.
work.mkdir(parents=True, exist_ok=True)
print(work)

# Show the current step while the library is working.
configure_logging()


### Understand the output

You should see the full workspace path. Open that folder to find the files created in these lessons. This path is a folder, not a forecast result.


Turning logging on does not run a forecast. Later library calls will show their steps. A stage count tells us how many steps have finished, not how much time is left.


## Step 2: Start the data engine

We will start a small local Spark session with two workers. Spark reads and checks the study data. UTC keeps date and time handling consistent.


In [ ]:
# Import the tools used in this step.
from pyspark.sql import SparkSession

# Use two local workers for this small example.
spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

### Understand the output

Spark may print startup messages. The cell should finish without an exception. No forecasts have been made yet.


## Step 3: Open the registry and saved run

We will load the same study registry and completed workflow used in earlier lessons.


In [ ]:
# Import the tools used in this step.
from autoforecast.study_registration import StudyRegistry
from autoforecast.study_workflow import load_forecast_workflow

# Open the registry described by our saved study settings.
registry = StudyRegistry.from_config(work / "study.yaml", spark=spark)

# Check and open the saved run without fitting again.
result = load_forecast_workflow((work / "latest-run.txt").read_text().strip())

### Understand the output

You may see verification messages while the saved files are checked. The saved registry and workflow have been opened; nothing has been published yet.


## Step 4: Connect and publish

We will read the connection string from the environment, create the example tables, and publish twice. Both result tables commit together. The final query lists the saved windows.


In [ ]:
# Import the PostgreSQL client and result destination.
import psycopg
from autoforecast.integrations.postgres import PostgresResults

# Read credentials from the environment; close the connection after use.
with psycopg.connect(os.environ["AUTOFORECAST_POSTGRES_DSN"], autocommit=True) as connection:
    # Use the public schema explicitly, keeping existing example tables.
    destination = PostgresResults(
        connection, schema="public", results="tutorial_results", forecasts="tutorial_forecasts"
    )
    destination.create_tables()
    # Parameter placeholders keep run identifiers separate from SQL text.
    where = "WHERE study_id = %s AND study_sha = %s AND run_id = %s"
    for attempt in range(2):
        publication = registry.publish_results(
            result.study_id, result.path, destination=destination
        )
        scope = (result.study_id, publication["study_sha256"], result.run_id)
        # Count only this run, even when the tables contain older runs.
        result_count = connection.execute(
            "SELECT COUNT(*) FROM public.tutorial_results " + where, scope
        ).fetchone()[0]
        forecast_count = connection.execute(
            "SELECT COUNT(*) FROM public.tutorial_forecasts " + where, scope
        ).fetchone()[0]
        print(
            "Publication",
            attempt + 1,
            publication["status"],
            "result rows:",
            result_count,
            "forecast rows:",
            forecast_count,
        )
    # Show each trial/window separately and use the same qualified table name.
    rows = connection.execute(
        "SELECT stage, trial_id, window_index, origin, horizon FROM public.tutorial_results "
        + where
        + " ORDER BY stage, trial_id, window_index, origin",
        scope,
    ).fetchall()
    for row in rows:
        print(row)

### Understand the output

Each publication should report `published`, 6 result rows and 84 forecast rows for this run. The second call should show the same counts, even if older runs are in the tables. The preview contains four tuning fits, validation and test. `trial_id` distinguishes the two tuning settings, and `window_index` is 0 or 1 for their earlier windows. These fields are empty (`null` or `None`) for validation and test. `origin` is the last training date; `horizon: 7` means seven future days. These six result rows summarize fits; the 84 forecast rows hold individual product/date predictions.


At INFO level, stage messages and a progress panel also show the work as it happens. The publication status and row counts described above is the result to keep. A completed-stage percentage counts steps; it does not predict how much time remains.


Both tables commit together. A failed write rolls back. Publication uses table locks to serialize writers, so a busy shared database may wait. The connection must be idle, with no pending transaction. Existing table schemas must match exactly; no tables are replaced. Drop the two tutorial tables yourself when you no longer need them.


## Step 5: Close this Spark session

We are finished with Spark in this notebook. Closing it frees resources before the next lesson.


In [ ]:
# Release Spark resources before moving to another notebook.
spark.stop()

### Understand the output

No output is expected. Spark is now closed. Start a fresh kernel when the next notebook asks you to.
